# Quadratic Score

Quadratic Discriminant Score and Cluster Quality Criteria
(Coraggio & Coretto, 2023)


"""
Coraggio & Coretto (2023) introduced two cluster-quality criteria derived 
from the quadratic discriminant score:

- Hard Score (Hₙ): assigns each point to the cluster whose quadratic score 
  is the highest (crisp assignment).
- Smooth Score (Tₙ): uses a soft weighting via a softmax transformation 
  of those scores, leading to smoother membership assignments.
"""

Quadratic Score
For a point xᵢ and clusters defined by parameters θₖ = (πₖ, μₖ, Σₖ):

$qs(xᵢ, θₖ) = log(πₖ) - 1/2 * log det(Σₖ) - 1/2 * (xᵢ - μₖ)ᵀ Σₖ⁻¹ (xᵢ - μₖ)$

Soft Membership (Weights)
The soft membership weight of point xᵢ in cluster k is:

$τₖ(xᵢ; θ) = exp(qs(xᵢ, θₖ)) / ∑ⱼ exp(qs(xᵢ, θⱼ))$ (i.e., a softmax transformation of the scores)

Smooth Score
The smooth score is defined as:

$Tₙ(θ) = (1/n) ∑ᵢ ∑ₖ τₖ(xᵢ; θ) · qs(xᵢ, θₖ)$

# Initialisation

In [25]:
## Set up
## Import Packages
import warnings
import os
import sys

import pandas as pd
import numpy as np
import re # Regular Expressions

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as patches

# import seaborn as sns
# import plotly.express as px

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score, fowlkes_mallows_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.model_selection import train_test_split
from sklearn.manifold import TSNE

import scipy
from scipy.spatial.distance import cdist
from scipy.stats import norm, skew, kurtosis, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from joblib import Parallel, delayed

# Time functions
import time
 
# Suppress Future Warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Suppress print() in functions
# import contextlib
import io

# Suppress prints
from contextlib import redirect_stdout

def suppress_print(func, *args, **kwargs):
    with open(os.devnull, "w") as fnull, redirect_stdout(fnull):
        return func(*args, **kwargs)

#Create Logs
import logging
logging.basicConfig(
    filename='debug_BQS.log',
    filemode='w',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True
)

# Data Storage Location:
data_storage = "/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/Quadratic Scores"

# Select Random Seed
RandomSeed = 42
print(f'Random seed is set to {RandomSeed}. Cluster algorithms and Neural Networks are optimization algorithms. Therefore the seed affects the initialization and so the model performance.')

Random seed is set to 42. Cluster algorithms and Neural Networks are optimization algorithms. Therefore the seed affects the initialization and so the model performance.


### Choose Settings

In [26]:
compute = True
save_results = False

if compute == True:
    test = False
    storage_location= '/Users/pattydegroot/Documents/Python/PsAID-Cluster-Analysis/PsAID_Pre-Processing & Inspection'
    algorithms = ['HAC_ward'] #['HAC_ward'] #['kmeans', 'HAC_ward', 'Spectral_Density']
    "n_bootstrap staat voor nu op b=100 omdat het computeren van de HAC wat meer tijd kost. Zou eigenlijk 1000 moeten zijn"

if not compute:
    test = True  # Uses iris dataset + kmeans
    algorithms = ['kmeans']
    "The example with kmeans implements the 'Hartigan & Wong kmeans algorithm, whilst the sklean kmeans applies Lloyd’s algorithm or Elkan if specifically defined"

## Import Dataset

In [27]:
if test:
    print('dataset = iris')
    dataset = sns.load_dataset("iris")
    columns_gp = ['sepal_length', 'sepal_width',  'petal_length',  'petal_width']
    print(dataset.head())
    dataset.info()

if compute:
    print('dataset = PsAID')
    ## Import Dataset
    dataset = pd.read_csv(f'{storage_location}/PsAID_Preprocessed.csv') # Import dataset
    dataset.head() # Check data
    
    ## Define Varlists
    columns_gp = [i for i in dataset.columns if re.match(r'^gp\d{2}$',i)]
    dataset = dataset[['respondentid', 'mm']+columns_gp]
    
    ## Dictionary
    domains_dict = {
        'gp01': 'Pain',
        'gp02': 'Fatigue',
        'gp03': 'Skin',
        'gp04': 'Work-recreational activities',
        'gp05': 'Bodily functioning',
        'gp06': 'Discomfort',
        'gp07': 'Sleep',
        'gp08': 'Coping',
        'gp09': 'Anxiety',
        'gp10': 'Shame',
        'gp11': 'Social activities',
        'gp12': 'Depression'
    }
    dataset.info()

dataset = PsAID
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   respondentid  5099 non-null   int64
 1   mm            5099 non-null   int64
 2   gp01          5099 non-null   int64
 3   gp02          5099 non-null   int64
 4   gp03          5099 non-null   int64
 5   gp04          5099 non-null   int64
 6   gp05          5099 non-null   int64
 7   gp06          5099 non-null   int64
 8   gp07          5099 non-null   int64
 9   gp08          5099 non-null   int64
 10  gp09          5099 non-null   int64
 11  gp10          5099 non-null   int64
 12  gp11          5099 non-null   int64
 13  gp12          5099 non-null   int64
dtypes: int64(14)
memory usage: 557.8 KB


# Bootstrap

## Cluster

### Kmeans

In [28]:
def kmeans_clustering(data, n_clusters, RandomSeed, byvar):   

    # Initialize and fit the KMeans model
    kmeans = KMeans(
        n_clusters=n_clusters,
        init='k-means++',
        n_init='auto',
        random_state= RandomSeed
    )
    clusterID = f'clusterID_KM_k{n_clusters}'
    data[clusterID] = kmeans.fit_predict(data[byvar]) #Assign cluster to each datapoint and store in the dataset dataframe

    del kmeans #Ensure the model is forgotten after each iteration
    return data, clusterID   

### Hierarchical

In [29]:
def HAC_ward_clustering(data, k):
    """
    Because our dataset is based on whole numbers [0-10], the HAC ward clustering is sensitive to the scale of the data. 
    There is higher similarity because of the lack of variance. Meaning that when scaled ties are assigned differently.
    """
    #link = 'ward'
    link = 'complete'
    #link = 'average'
    distance_metric = 'euclidean' # Ward can only be performed with euclidean distance
    #distance_metric = 'manhattan'
    # thresholds = {2:400, 3:230, 5:130, 7:100} # Ward
    # thresholds = {2:33, 3:26, 4:23}   # Complete

    if distance_metric == 'manhattan':
        linkage_matrix_distance = 'cityblock'
        distance_suffix = 'L1'
    else:
        linkage_matrix_distance = distance_metric
        distance_suffix = ''
    
    # Run agglomeration algorithm
    HAC_Ward = AgglomerativeClustering(n_clusters=10, linkage = link, metric = distance_metric).fit(data[columns_gp])
    
    # Create linkage matrix
    linkage_matrix = linkage(data[columns_gp], method=link, metric = linkage_matrix_distance)
    
    # Assign Clusters
    cluster_assignment = fcluster(linkage_matrix, k, criterion = 'maxclust')-1 #flcuster outputs clusters as 1-K, but we need 0-K for consistency across techniques
    #cluster_assignment = fcluster(linkage_matrix, t=thresholds[k], criterion = 'distance')-1 #flcuster outputs clusters as 1-K, but we need 0-K for consistency across techniques
    n_clusters = len(np.unique(cluster_assignment))

    """ Cuts can produce different n_clusters, therefor use K. Kan be that K=3 has 4 clusters """
    clusterID = f'clusterID_HAC{link}_k{k}' #
    
    data[clusterID] = cluster_assignment
    # print(data.head())

    # PRINTS FOR DEBUGGING
    print(f'k:{n_clusters}')
    print(f'sizes: {data[clusterID].value_counts().sort_index().values}')
    print(clusterID)
    
    # # Plot Dendrogram
    # plt.figure(figsize=(10,7))
    # dendrogram(linkage_matrix, labels=data[columns_gp].index, leaf_rotation=90, leaf_font_size=8)
    # plt.title(f'Hierarchical Clustering Dendrogram, link {link}')
    # plt.xlabel("Sample Index")
    # plt.ylabel(f'Distance {distance_metric}')
    # for t in [2, 3, 5, 7]:
    #     plt.axhline(y=thresholds[t])
    # plt.show()

    del HAC_Ward
    return data, clusterID

### Spectral

In [30]:
def scale_data(data, scaling):
    """ ATTENTION: THE SCALE OF THE DATA AFFECTS THE QS SCORE THROUGH THE LOGDET(), This is solved when computing the BQS by only returning the cluster assignments and merge it with the original data"""
    if scaling == 'div10':
         data_scaled = data/10
    elif scaling == 'StandardScaled':
         scaler = StandardScaler()
         data_scaled = pd.DataFrame(
            scaler.fit_transform(data[columns_gp]),      # Only values are transformed
            columns=columns_gp,            # Keep original column names
            index=data[columns_gp].index                 # Optional: keep original index
         )

    return data_scaled

def spectral_clustering(data_scaled, settings, RandomSeed):
    settings = settings.copy()
    scaling = settings.pop('scaling') # Remove item from dict and return value

    # Initialize and fit the KMeans model
    spec_clust = SpectralClustering(**settings,
                    # n_clusters=k, 
                    # affinity = aff,
                    # gamma = gamma,
                    # assign_labels=ass,
                    random_state=RandomSeed
    )

    # Fit the model and get the predicted labels
    clusterID = f'clusterID_{scaling}_{settings['affinity']}{settings['gamma']}_{settings['assign_labels']}_k{settings['n_clusters']}'
    cluster_assignment = spec_clust.fit_predict(data_scaled)

    # # Save the embedding layer
    # # Extract the spectral embedding used by SpectralClustering()
    # embedding = spectral_embedding(
    #     adjacency=spec_clust.affinity_matrix_,
    #     n_components=cluster,
    #     random_state=RandomSeed
    # )

    del spec_clust
    return clusterID, cluster_assignment

def run_spectral_clustering(data, settings, byvar, RandomSeed):

    # Scale data for easy computation
    data_scaled = scale_data(data[byvar], settings['scaling'])

    # Spectral Projection + Clustering
    clusterID, cluster_assignment = spectral_clustering(data_scaled, settings, RandomSeed)

    # Add clusters to dataset
    data = data.copy()
    data[clusterID] = cluster_assignment

    return data, clusterID

In [31]:
for algorithm in algorithms:
    print(algorithm)
    if algorithm == 'kmeans':
        for k in range(2,11): 
            dataset, clusterID = kmeans_clustering(dataset, k, RandomSeed, columns_gp)
            #clusterID
    if algorithm == 'HAC_ward':
        for k in [2,3,4, 5,7]:
            dataset, clusterID = HAC_ward_clustering(dataset, k)
    if algorithm == 'Spectral_Density': 
        spectral_settings = { # K3 DIV 10
                              "run_K3A":{"scaling": 'div10', "n_clusters": 3 ,"affinity": 'rbf', "gamma": 3.9 , "assign_labels": 'cluster_qr'},
                              "run_K3B":{"scaling": 'div10', "n_clusters": 4 ,"affinity": 'rbf', "gamma": 3.9, "assign_labels": 'cluster_qr'},
                              "run_K3C":{"scaling": 'div10', "n_clusters": 5 ,"affinity": 'rbf', "gamma": 3.9, "assign_labels": 'cluster_qr'},
                              "run_K3D":{"scaling": 'div10', "n_clusters": 6 ,"affinity": 'rbf', "gamma": 5.1, "assign_labels": 'cluster_qr'},
                              "run_K3E":{"scaling": 'div10', "n_clusters": 7 ,"affinity": 'rbf', "gamma": 5.1, "assign_labels": 'cluster_qr'},
                              }
                             
        for sett in spectral_settings:
             dataset, clusterID = run_spectral_clustering(dataset, spectral_settings[sett], columns_gp, RandomSeed)

dataset

HAC_ward
k:2
sizes: [ 383 4716]
clusterID_HACcomplete_k2
k:3
sizes: [ 383 3697 1019]
clusterID_HACcomplete_k3
k:4
sizes: [ 383 3697  179  840]
clusterID_HACcomplete_k4
k:5
sizes: [ 383 3697  179   87  753]
clusterID_HACcomplete_k5
k:7
sizes: [ 383  673 2420  604  179   87  753]
clusterID_HACcomplete_k7


,respondentid,mm,gp01,gp02,gp03,gp04,gp05,gp06,gp07,gp08,gp09,gp10,gp11,gp12,clusterID_HACcomplete_k2,clusterID_HACcomplete_k3,clusterID_HACcomplete_k4,clusterID_HACcomplete_k5,clusterID_HACcomplete_k7
0,78195329,36,2,2,2,2,2,0,1,0,0,0,0,0,1,1,1,1,2
1,10862915,6,1,6,2,1,1,1,1,1,1,0,0,2,1,1,1,1,1
2,29370588,18,6,7,5,6,4,6,7,5,6,2,8,5,1,2,2,2,4
3,87648336,18,0,4,5,0,0,0,0,0,0,1,0,0,1,1,1,1,3
4,42677901,84,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5094,48031533,0,3,4,1,2,2,3,2,2,1,0,2,1,1,1,1,1,2
5095,71662504,9,3,3,0,6,6,3,0,6,6,0,1,1,1,1,1,1,3
5096,63159202,18,7,8,10,8,8,10,10,10,10,10,9,10,0,0,0,0,0
5097,36477829,36,7,6,0,6,6,6,6,6,6,4,6,4,1,2,3,4,6


# Compute QS

The quadratic smooth criterion looks for a compromise between the best approximation of the finite mixture density ($\psi$)

1. Compute cluster representations ($\theta_{k}$)
   - $\pi_{k}$ = size --> fraction of points belonging to cluster k
   - $\mu_{k}$ = location --> cluster centers
   - $\Sigma_{k}$ = scatter --> pos.def scatter matrix, which coincides / is proportional to cov. matrix <br>
       If the cluster has very few points OR some features are linearly dependent the matrix can be singular and thus not invertible.

2. Compute the quadratic score ($qs$) for each point in each cluster. <br>
    The quadratic score is a measure of fit of $X$ into the $k^{th}$ cluster according to $\theta_{k}$

4. Compute the Quadratic Smooth Score ($T_{n}(\theta)$) <br>
    Smooth weighting ($\tau$) is obtained by normalizing the quadratic scores with softmax transformation. Other weighting schemes are possible, but softmax guarantees some form of optimality for Gaussian clusters. 
    - $T_n(\theta) = \frac{1}{n} \sum_{i=1}^{n} \sum_{k=1}^{K} \tau_k(x_i; \theta) \, qs(x_i; \theta_k)$ ^ $\tau_k(x_i; \theta) = \frac{\exp(qs(x_i; \theta_k))}{\sum_{j=1}^{K} \exp(qs(x_i; \theta_j))}$ <br>

In [32]:
def compute_QS(data, byvar, cluster_col):
   
    n,p = data.shape  # Num_rows, Num_columns
    labels = data[cluster_col] # Cluster assignments
    k = len(labels.unique()) # Number of clusters
    print(f'K:{k}')

    print(f'\n Computing QS for {cluster_col}')

    # Extract parameters
    cluster_size = labels.value_counts().sort_index().values # Array of cluster sizes
    cluster_sizefraction = labels.value_counts().sort_index().values/n # Fraction of the sample belonging to each cluster
    cluster_centers = data[byvar + [cluster_col]].groupby(cluster_col).mean()  # Centroid vector
    print('cluster_size:', cluster_size)

    # ---- validity check ----
    # Check for singular mixture components. If <2 the cov matrix is undefined/dof<=0 -->  If singular the $inv(Sigma_k)$ can not be computed because det(Sigma_k) = 0.
    # Thus can't calculate Tn
    if np.any(cluster_size <2):
        Tn = np.nan
        tau = np.nan
        if len(labels.unique()) != int(re.search(r'k(\d+)', cluster_col).group(1)): 
            comments = f'k ≠ label & cluster {np.where(cluster_size < 2)[0].tolist()} invalid'
        else:    
            comments = f'cluster {np.where(cluster_size < 2)[0].tolist()} invalid'
        return Tn, tau, comments, cluster_size
 
    # Compute Quadratic Score qs
    X = data[byvar].to_numpy() #Convert psaid data to numpy array
    qs_matrix = np.full((n, k), np.nan)  # Reserve space for qs_matrix
    
    for k in sorted(labels.unique()):

        pi_k = cluster_sizefraction[k] # Fraction of cluster k
        mu_k = cluster_centers.loc[k].to_numpy() # Centers of cluster k

        # Compute scatter/covariance
        cluster_data = data.loc[labels == k, byvar]
        Sig_k = np.cov(cluster_data, rowvar=False, ddof=1) # Compute sample scatter = covariance matrix of cluster k

        # Check for linear dependencies --> Compute Rank of sample 
        # If the cov matrix is singular --> Ridge regularization to make invertible
        if np.linalg.matrix_rank(Sig_k) < len(byvar): 
            Sig_k += 1e-6 * np.eye(len(byvar))

        # Compute the quadratic term = 1/2 * (xᵢ - μₖ)ᵀ Σₖ⁻¹ (xᵢ - μₖ)
        diff = X - mu_k # Difference Vectors
        inv_Sig = np.linalg.inv(Sig_k) #Log DEeterminant
        quad = np.einsum('ij,jk,ik->i', diff, inv_Sig, diff) # Quadratic term = Mahalanobis distance squared # Einsum lets you define custom summation patterns

        # Compute Log Determinant of Sig_k
        # By using slogdet and ignoring sign, Sig_k is assumed positive-definite, which is usually fine if you regularize
        sign, logdet = np.linalg.slogdet(Sig_k)
 
        # Compute the Quadratice Scores for all observations in cluster k
        qs = np.log(pi_k)-0.5*logdet-0.5*quad
        qs_matrix[:,k] = qs # Store all qs in one matrix

    
    # Compute the Quadratic Smooth Score (QS) = Tₙ(θ) = (1/n) ∑ᵢ ∑ₖ τₖ(xᵢ; θ) · qs(xᵢ, θₖ)
    # Compute the softmax weights with Softmax transformation --> τₖ(xᵢ; θ) = τₖ(xᵢ; θ) = exp(qs(xᵢ, θₖ)) / ∑ⱼ exp(qs(xᵢ, θⱼ))
    # Softmax ensures the scores are "probability like"
    max_qs = np.max(qs_matrix, axis=1, keepdims=True)  # For numerical stability, subtract the max per row
    exp_qs = np.exp(qs_matrix - max_qs)
    tau = exp_qs / exp_qs.sum(axis=1, keepdims=True)  # Softmax weights
    print("tau:",tau)

    # Compute  QS
    Tn = np.mean(np.sum(tau * qs_matrix, axis=1))
    print("QS:",Tn)

    # Keep track of skipped clusters because too cluster size <2              
    if len(labels.unique()) != int(re.search(r'k(\d+)', cluster_col).group(1)): 
            comments = f'k ≠ label'
    else: 
            comments = ""

    return Tn, tau, comments, cluster_size

In [33]:
## Computed with Unscaled data
cluster_columns = [i for i in dataset.columns if re.match(r'^clusterID_',i)]

QS = []
for cluster in cluster_columns:
    Tn, _, comments, cluster_size = compute_QS(dataset[columns_gp+[cluster]], columns_gp, cluster)
    QS.append({'K': cluster,'QS': Tn, 'QS_comment': comments, 'Cluster Size': cluster_size})

QS_df = pd.DataFrame(QS)

if save_results:
    #Mark dataset with computation date.time
    timestamp = datetime.now().strftime("%Y%m%d_%H:%M:%S")
    QS_df = pd.concat(QS_df, ignore_index=True)
    QS_df['timestamp'] = timestamp

    # Store
    file_name = f'PsAID_QS_{timestamp}'
    
    # Ensure the folder exists
    os.makedirs(data_storage, exist_ok=True)
    
    # Save
    QS_df.to_csv(f'{data_storage}/{file_name}.csv', index=False)
    print(f' Dataset saved to {data_storage}/{file_name}.csv')

QS_df

K:2

 Computing QS for clusterID_HACcomplete_k2
cluster_size: [ 383 4716]
tau: [[3.65789077e-13 1.00000000e+00]
 [2.92120447e-10 1.00000000e+00]
 [2.65072723e-01 7.34927277e-01]
 ...
 [9.99999991e-01 8.53705084e-09]
 [7.96488525e-01 2.03511475e-01]
 [5.89228296e-15 1.00000000e+00]]
QS: -11.368644875604362
K:3

 Computing QS for clusterID_HACcomplete_k3
cluster_size: [ 383 3697 1019]
tau: [[6.92539794e-14 9.99999964e-01 3.56690171e-08]
 [8.62267727e-11 9.99998340e-01 1.66027337e-06]
 [4.18020935e-02 2.53901825e-08 9.58197881e-01]
 ...
 [9.99999620e-01 1.46360549e-27 3.79971023e-07]
 [6.37631597e-01 3.39186428e-06 3.62365011e-01]
 [2.05979870e-15 9.99981440e-01 1.85600981e-05]]
QS: -10.384874190785968
K:4

 Computing QS for clusterID_HACcomplete_k4
cluster_size: [ 383 3697  179  840]
tau: [[6.92539769e-14 9.99999927e-01 4.18799910e-17 7.25739850e-08]
 [8.62267427e-11 9.99997992e-01 4.94569980e-13 2.00823675e-06]
 [4.06608220e-02 2.46969854e-08 4.20013783e-01 5.39325370e-01]
 ...
 [1.0000

,K,QS,QS_comment,Cluster Size
0,clusterID_HACcomplete_k2,-11.368645,,"[383, 4716]"
1,clusterID_HACcomplete_k3,-10.384874,,"[383, 3697, 1019]"
2,clusterID_HACcomplete_k4,-10.401646,,"[383, 3697, 179, 840]"
3,clusterID_HACcomplete_k5,-10.402623,,"[383, 3697, 179, 87, 753]"
4,clusterID_HACcomplete_k7,-9.318364,,"[383, 673, 2420, 604, 179, 87, 753]"


# Compute BQS
Algorithm 2 - page 10 + Supplement S5

**Compute BQS**

1. Bootstrap sampling ($b$=1000) <br>
   This method involves randomly sampling with replacement from the original dataset to create multiple smaller samples. It is commonly used to estimate the distribution of a statistic. One of its most common uses is to estimate confidence intervals when the underlying distribution is unknown or when sample sizes are small.
   - Classical Efrons non-parametric bootstrap idea (use case: Estimating the variability of statistics like the mean, median, regression coefficients)
   - Use 1000 bootstraps for real datasets

2. Compute clusters with each bootstrap sample ($\theta^{b}$)

3. Compute the quadratic score as explained previously ($S^{b}$ = sample_QS)

4. Compute the mean quadratic score over all $b$ ($W_{n}$)

5. Standardize $W_{n}$ ($R_{n}$)

6. Compute the 95% CI ($L_{n}$, $U_{n}$)

**Select the best clutering solution:** <br> 
Rather than selecting cluster configurations achieving the largest estimated W. We propose to select BQS = argmax($L_{n}$).
  


In [34]:
def compute_ARI(OG_data, sample_data, byvar, cluster_col):
    OG_labels = OG_data[cluster_col] # Reference cluster assignments
    X_OG = OG_data[byvar].to_numpy()
    OG_n = OG_data.shape[0]
    sample_n = sample_data.shape[0]

    # Extract bootstrap cluster descriptions
    sample_labels = sample_data[cluster_col] # Cluster assignments
    k = len(sample_labels.unique()) # Number of clusters
    cluster_centers = sample_data[byvar + [cluster_col]].groupby(cluster_col).mean()  # Centroid vector
    cluster_size = sample_labels.value_counts().sort_index().values # Array of cluster sizes
    print("cluster_size:", cluster_size)
    cluster_sizefraction = sample_labels.value_counts().sort_index().values/sample_n # Fraction of the sample belonging to each cluster

    # ---- validity check ----
    # Check for singular mixture components. If <2 the cov matrix is undefined/dof<=0 -->  If singular the $inv(Sigma_k)$ can not be computed because det(Sigma_k) = 0.
    # Thus can't calculate tau
    if np.any(cluster_size <2):
        sample_ARI = np.nan
        return sample_ARI

    # Reserve storage space
    qs_matrix = np.full((OG_n, k), np.nan)
   # qs_matrix = np.zeros((OG_n, k))
   # print("qs matrix:", np.shape(qs_matrix))
    
    # Probabilites of original data belonging to bootstrap cluster k
    for k in sorted(sample_labels.unique()):

        cluster_data = sample_data.loc[sample_labels == k, byvar]
        
        # Compute scatter/covariance
        Sig_k = np.cov(cluster_data, rowvar=False, ddof=1) # Compute sample scatter = covariance matrix of cluster k
        if np.linalg.matrix_rank(Sig_k) < len(byvar): # If not full rank
            Sig_k += 1e-6 * np.eye(len(byvar))        # Ridge Regularization

        mu_k = cluster_centers.loc[k].to_numpy() # Centers of cluster k
        pi_k = cluster_sizefraction[k] # Cluster size fraction

        # Compute the quadratic term = 1/2 * (xᵢ - μₖ)ᵀ Σₖ⁻¹ (xᵢ - μₖ)
        diff = X_OG - mu_k # Difference Vectors
        inv_Sig = np.linalg.inv(Sig_k) #Log DEeterminant
        quad = np.einsum('ij,jk,ik->i', diff, inv_Sig, diff) # Quadratic term = Mahalanobis distance squared # Einsum lets you define custom summation patterns

        # Compute Log Determinant of Sig_k
        # By using slogdet and ignoring sign, Sig_k is assumed positive-definite, which is usually fine if you regularize
        sign, logdet = np.linalg.slogdet(Sig_k)
 
        # Compute the Quadratice Scores for all observations in cluster k
        qs = np.log(pi_k)-0.5*logdet-0.5*quad
        qs_matrix[:,k] = qs # Store all qs in one matrix

    # Softmax transformation to ensure scores are 'probability like'
    max_qs = np.max(qs_matrix, axis=1, keepdims=True)  # For numerical stability, subtract the max per row
    exp_qs = np.exp(qs_matrix - max_qs)
    tau = exp_qs / exp_qs.sum(axis=1, keepdims=True)  # Softmax weights = probabilities

    # Assign OG_data to the most likely bootstrap cluster
    projected_labels = tau.argmax(axis=1)

    # Compute Ari against original labels
    sample_ARI = adjusted_rand_score(OG_labels, projected_labels)

    return sample_ARI

In [35]:
def bootstrap_iteration(b, data, byvar, algorithm, s, randomstate):
    import os, logging
    logging.basicConfig(
        filename='debug_BQS.log',
        filemode='a',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        force=True
    )
    
    n = data.shape[0]
    OG_data = data.copy()

    # Sample with replacement
    sample_idx = np.random.choice(n, n, replace=True)
    sample_data = data.iloc[sample_idx].reset_index(drop=True)

    # Run clustering
    if algorithm == 'kmeans':
        n_clusters = s
        sample_data, clusterID = kmeans_clustering(sample_data, n_clusters, randomstate, byvar)

    elif algorithm == 'HAC_ward':
            n_clusters = s
            sample_data, clusterID = HAC_ward_clustering(sample_data, n_clusters)
            logging.info(f"{algorithm}_{s}, b:{b}, computed_k: {len(np.unique(sample_data[clusterID]))}")

    elif algorithm == 'Spectral_Density':
        setting = s
        sample_data, clusterID = run_spectral_clustering(sample_data, setting, byvar, randomstate)

    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")

    # Compute sample QS
    sample_QS, sample_tau, sample_comment, _ = suppress_print(compute_QS, sample_data, byvar, clusterID)
    # Turn on prints for debugging
    #print('computing QS')
    #sample_QS, sample_tau, sample_comment = compute_QS(sample_data, byvar, clusterID)  # For debugging

    # Compute ARI
    sample_ARI = suppress_print(compute_ARI, OG_data[byvar+[clusterID]], sample_data[byvar+[clusterID]], byvar, clusterID)
    # Turn on prints for debugging
    # Turn on prints for debugging
    #sample_ARI = compute_ARI(OG_data[byvar+[clusterID]], sample_data[byvar+[clusterID]], byvar, clusterID)

    return sample_QS, sample_comment, clusterID, sample_ARI

In [36]:
def compute_BQS_ARI_inParallel(data, byvar, algorithm, s, n_bootstrap=1000, randomstate=RandomSeed, n_jobs=-1):
    start = time.time()
    
    # Run bootstraps in parallel
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(  #For debuggin use backend = threading", otherwise "loky" to suppress prints
        delayed(bootstrap_iteration)(b, data, byvar, algorithm, s, randomstate)
        for b in range(n_bootstrap)
    )
    
    # Unpack results
    qs_list, comment_list, clusterIDs, ARI_list = zip(*results)
    clusterID = clusterIDs[-1]  # use last clusterID (same as before)

    qs_array = np.array(qs_list) # Convert list to numpy array
  
    # ---- validity check ----
    # If +> 25% of the runs are invalid, discard the discard method.
    n_nan = np.sum(np.isnan(qs_array))  # Count missing sample_qs values
    # n_nonempty_c = sum([1 for c in comment_list if c.strip() != ""])  # based on  comments 
    proportion_empty = n_nan / n_bootstrap * 100   # Convert # missing values to %
    if proportion_empty > 24: 
        BQS = np.nan
        results_dict = {
            'K': clusterID,
            'ARI': np.nan,
            'QS_mean': np.nan,
            'L_ci': np.nan,
            'U_ci': np.nan,
            # 'all_QS': qs_array,
            '%-discarded': proportion_empty,
            'duration': np.nan,
            'b': n_bootstraps
        }
        return clusterID, BQS, results_dict

    # Verify for HAC how many times the clusterID does not match the #clusters computed
    logging.info(comment_list)
    label_k_mismatch_pattern = r'k\s*≠\s*label'
    label_k_mismatch = sum(bool(re.search(label_k_mismatch_pattern, c)) for c in comment_list)
    proportion_mismatch = label_k_mismatch / n_bootstrap * 100   # Convert # mismatches to % 
    logging.info(f'{algorithm}_{s} - b: {n_bootstrap} | #_mismatches: {label_k_mismatch}| proportion_mismatch: {proportion_mismatch}')
        
    # Compute BQS
    qs_valid = qs_array[~np.isnan(qs_array)]
    W = qs_valid.mean()
    R = np.sqrt(data.shape[0]) * (qs_valid - W)
    alpha = 0.05
    L = np.quantile(R, alpha/2) 
    U = np.quantile(R, 1 - alpha/2)

    BQS = L

    # Average ARI
    ARI_array = np.array(ARI_list)
    ARI_mean = np.nanmean(ARI_array)

    results_dict = {
        'K': clusterID,
        'ARI': ARI_mean,
        'QS_mean': W,
        'L_ci': L,
        'U_ci': U,
       # 'all_QS': qs_array,
        '%-discarded': proportion_empty,
        '%-mismatch':proportion_mismatch,
        'duration': f'{time.time()-start:.2f}s',
        'b': n_bootstrap
    }

    print(pd.DataFrame([results_dict])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', '%-mismatch', 'duration']])

    return clusterID, round(BQS,5), results_dict

In [37]:
BQS = []
bqs_list = []

for algorithm in algorithms:
    if algorithm == 'kmeans':
        for n_clust in range(2,10):
            clusterID, qs_bootstrapped, bqs_results = compute_BQS_ARI_inParallel(dataset, columns_gp, algorithm, s = n_clust)
            BQS.append({'K': clusterID, 'BQS':qs_bootstrapped})
            bqs_list.append(pd.DataFrame([bqs_results])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', '%-mismatch', 'duration', 'b']])
    
    if algorithm == 'HAC_ward':
            for n_clust in [2,3,4, 5,7]:
                clusterID, qs_bootstrapped, bqs_results= compute_BQS_ARI_inParallel(dataset, columns_gp, algorithm, s = n_clust)
                BQS.append({'K': clusterID, 'BQS':qs_bootstrapped})
                bqs_list.append(pd.DataFrame([bqs_results])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', '%-mismatch', 'duration', 'b']])

    if algorithm == 'Spectral_Density': 
        for sett in spectral_settings:
            clusterID, qs_bootstrapped, bqs_results = compute_BQS_ARI_inParallel(dataset, columns_gp, algorithm, s = spectral_settings[sett])
            BQS.append({'K': clusterID, 'BQS':qs_bootstrapped})
            bqs_list.append(pd.DataFrame([bqs_results])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', '%-mismatch', 'duration', 'b']])

os.system('say "BQS Computed!"')
print('Done computing BQS')
BQS_df = pd.concat(bqs_list, ignore_index=True)

if save_results: 
    #Mark dataset with computation date.time
    timestamp = datetime.now().strftime("%Y%m%d_%H:%M:%S")
    BQS_df = pd.concat(bqs_list, ignore_index=True)
    BQS_df['timestamp'] = timestamp
    
    # Store
    file_name = f'PsAID_BQS_{timestamp}'
    
    # Ensure the folder exists
    os.makedirs(data_storage, exist_ok=True)
    
    # Save
    BQS_df.to_csv(f'{data_storage}/{file_name}.csv', index=False)
    print(f' Dataset saved to {data_storage}/{file_name}.csv')

BQS_df

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   12.4s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   28.1s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:   48.8s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed: 15.1min
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed: 15.5min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


                          K       ARI   QS_mean       L_ci       U_ci  \
0  clusterID_HACcomplete_k2  0.439034 -10.99546 -33.953417  44.759848   

   %-discarded  %-mismatch duration  
0          0.0         0.0  927.36s  


[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   10.2s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   25.7s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:   46.1s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed:  2.4min
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:  2.8min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


                          K       ARI    QS_mean       L_ci       U_ci  \
0  clusterID_HACcomplete_k3  0.653186 -10.428633 -39.732411  49.786513   

   %-discarded  %-mismatch duration  
0          0.0         0.1  168.42s  


[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    1.4s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    9.6s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   25.7s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:   47.3s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed:  2.5min
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed: 36.6min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


...

[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed:  2.7min
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:  3.1min finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


                          K      ARI    QS_mean       L_ci       U_ci  \
0  clusterID_HACcomplete_k5  0.60242 -10.004237 -39.197139  43.672357   

   %-discarded  %-mismatch duration  
0          0.0         0.5  184.67s  


[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    1.3s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    9.8s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:   26.4s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:   47.4s
[Parallel(n_jobs=-1)]: Done 434 tasks      | elapsed:  8.9min
[Parallel(n_jobs=-1)]: Done 632 tasks      | elapsed:  9.5min
[Parallel(n_jobs=-1)]: Done 866 tasks      | elapsed: 12.9min
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed: 13.8min finished


                          K       ARI   QS_mean       L_ci       U_ci  \
0  clusterID_HACcomplete_k7  0.453659 -9.691443 -32.581285  34.485728   

   %-discarded  %-mismatch duration  
0          0.0         2.4  826.13s  
Done computing BQS


,K,ARI,QS_mean,L_ci,U_ci,%-discarded,%-mismatch,duration,b
0,clusterID_HACcomplete_k2,0.439034,-10.995460,-33.953417,44.759848,0.0,0.0,927.36s,1000
1,clusterID_HACcomplete_k3,0.653186,-10.428633,-39.732411,49.786513,0.0,0.1,168.42s,1000
2,clusterID_HACcomplete_k4,0.631826,-10.206347,-38.558203,47.642731,0.0,0.4,2194.70s,1000
3,clusterID_HACcomplete_k5,0.602420,-10.004237,-39.197139,43.672357,0.0,0.5,184.67s,1000
4,clusterID_HACcomplete_k7,0.453659,-9.691443,-32.581285,34.485728,0.0,2.4,826.13s,1000


# APPENDIX

## Compute BQS with Scaled Data

In [67]:
scales = ['unscaled', 'div10', 'StandardScaled']

if algorithm == 'kmeans':
    for k in range(2,10): 
        dataset, clusterID = kmeans_clustering(dataset, k, RandomSeed, columns_gp)
        
    for scaling in scales: 
            data_scaled = dataset
            if scaling == 'unscaled':
                data_scaled
            elif scaling == 'div10':
                data_scaled[columns_gp] = dataset[columns_gp]/10
            elif scaling == 'StandardScaled':
                scaler = StandardScaler()
                data_scaled[columns_gp] = pd.DataFrame(
                    scaler.fit_transform(dataset[columns_gp]),      # Only values are transformed
                    columns=columns_gp,            # Keep original column names
                    index=dataset[columns_gp].index                 # Optional: keep original index
                )
            else:   raise ValueError(f"Unknown scaling: {scaling}")

            data_scaled, clusterID = kmeans_clustering(data_scaled, k, RandomSeed, columns_gp)

            print(scaling)
            for n_clust in range(2,10):
                clusterID, qs_bootstrapped, bqs_results = compute_BQS_ARI_inParallel(data_scaled, columns_gp, algorithm, s = n_clust)
                BQS.append({'K': clusterID, 'BQS':qs_bootstrapped})
                bqs = pd.DataFrame([bqs_results])[['K', 'ARI', 'QS_mean', 'L_ci', 'U_ci', '%-discarded', 'duration', 'b']]
                bqs.insert(0, 'scaling', scaling)
                bqs_list.append(bqs)

    os.system('say "BQS Scaling test Computed!"')
    pd.concat(bqs_list, ignore_index=True)